In [ ]:
import requests
import pandas as pd
from tqdm import tqdm
import time

# ---------------------------
# 1️⃣ Récupérer les artistes d'un genre
# ---------------------------
genre_id = 165  # Afrobeat
artists_list = []

url = f"https://api.deezer.com/genre/{genre_id}/artists"
while url:
    data = requests.get(url).json()
    for a in data.get("data", []):
        # Requête supplémentaire pour avoir nb_fan et link
        r = requests.get(f"https://api.deezer.com/artist/{a['id']}").json()
        artists_list.append({
            "id": r["id"],
            "name": r["name"],
            "link": r.get("link", ""),
            "nb_fan": r.get("nb_fan", 0),
            "picture": r.get("picture_medium", "")
        })
        time.sleep(0.1)  # éviter de surcharger l'API
    url = data.get("next")

artists_df = pd.DataFrame(artists_list)
artists_df.to_csv("data/artists.csv", index=False, encoding="utf-8")
print(f" {len(artists_df)} artistes enregistrés dans artists.csv")

# ---------------------------
# 2️⃣ Récupérer les albums de ces artistes
# ---------------------------
albums_list = []

for artist in tqdm(artists_list):
    url = f"https://api.deezer.com/artist/{artist['id']}/albums"
    while url:
        data = requests.get(url).json()
        for alb in data.get("data", []):
            albums_list.append({
                "artist_id": artist["id"],
                "artist_name": artist["name"],
                "album_id": alb["id"],
                "album_title": alb["title"],
                "release_date": alb.get("release_date", ""),
                "nb_tracks": alb.get("nb_tracks", 0),
                "cover": alb.get("cover_medium", ""),
                "link": alb.get("link", "")
            })
        url = data.get("next")
        time.sleep(0.1)

albums_df = pd.DataFrame(albums_list)
albums_df.to_csv("data/albums.csv", index=False, encoding="utf-8")
print(f"💾 {len(albums_df)} albums enregistrés dans albums.csv")

# ---------------------------
# 3️⃣ Récupérer les tracks de ces albums
# ---------------------------
tracks_list = []

for album in tqdm(albums_list):
    url = f"https://api.deezer.com/album/{album['album_id']}/tracks"
    while url:
        data = requests.get(url).json()
        for t in data.get("data", []):
            tracks_list.append({
                "album_id": album["album_id"],
                "track_id": t["id"],
                "track_title": t["title"],
                "duration_sec": t["duration"],
                "rank": t["rank"],
                "preview_url": t.get("preview", "")
            })
        url = data.get("next")
        time.sleep(0.1)

tracks_df = pd.DataFrame(tracks_list)
tracks_df.to_csv("data/tracks.csv", index=False, encoding="utf-8")
print(f"{len(tracks_df)} tracks enregistrés dans tracks.csv")


 48 artistes enregistrés dans artists.csv


100%|██████████| 48/48 [00:32<00:00,  1.48it/s]


💾 2475 albums enregistrés dans albums.csv


100%|██████████| 2475/2475 [09:21<00:00,  4.41it/s]

15989 tracks enregistrés dans tracks.csv
